In [1]:
!pip install -q datasets

In [11]:
!ls /kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw

audioldm2  MusicGen_medium  musicldm  mustango	stable_audio_open


In [12]:
import os
R='/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw'
for g in ['audioldm2','MusicGen_medium','musicldm','mustango','stable_audio_open']:
    print(g, len([f for f in os.listdir(f'{R}/{g}') if f.endswith('.wav')]))

audioldm2 5521
MusicGen_medium 5521
musicldm 5521
mustango 5521
stable_audio_open 5521


In [2]:
from datasets import load_dataset
import pandas as pd

ds   = load_dataset("CLAPv2/MusicCaps", split="train")
meta = load_dataset("google/MusicCaps", split="train").to_pandas()

# CLAPv2 has the audio but dropped the YouTube ID, so we recover it by
# joining on the caption. Captions are long free-text, so the match is unique.
real   = ds.remove_columns(['audio']).to_pandas()
merged = real.merge(meta[['ytid','caption']], on='caption', how='left')
print('rows:', len(merged), '| matched:', merged['ytid'].notna().sum(),
      '| unique:', merged['ytid'].nunique())

musiccaps_part_0.parquet: reconstructing file:   0%|          |  0.00B /  927MB            

musiccaps_part_0.parquet: downloading bytes:           |  0.00B            

musiccaps_part_1.parquet: reconstructing file:   0%|          |  0.00B /  936MB            

musiccaps_part_1.parquet: downloading bytes:           |  0.00B            

musiccaps_part_10.parquet: reconstructing file:   0%|          |  0.00B /  628MB            

...

README.md:   0%|          | 0.00/5.06k [00:00<?, ?B/s]

musiccaps-public.csv: reconstructing file:   0%|          |  0.00B / 2.94MB            

musiccaps-public.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5521 [00:00<?, ? examples/s]

rows: 5352 | matched: 5352 | unique: 5352


In [3]:
import os, soundfile as sf, torchaudio, torch

OUT = '/kaggle/working/real'
os.makedirs(OUT, exist_ok=True)
TARGET_SR = 16000
ytids = merged['ytid'].tolist()

# Authentic audio is 48 kHz stereo s16; synthetic is 16 kHz mono float32.
# Left unmatched, sample rate alone would separate the classes perfectly
# and no model would ever look at the music. This is the control.
written = 0
for i in range(len(ds)):
    s = ds[i]['audio'].get_all_samples()
    wav, sr = s.data, s.sample_rate
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    sf.write(f'{OUT}/{ytids[i]}.wav',
             wav.squeeze(0).to(torch.float32).numpy(), TARGET_SR, subtype='FLOAT')
    written += 1
print('written:', written)

written: 5352


In [4]:
import soundfile as sf, os
SYN = '/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw'
f = os.listdir('/kaggle/working/real')[0]
g = os.listdir(f'{SYN}/audioldm2')[0]
i = sf.info(f'/kaggle/working/real/{f}')
j = sf.info(f'{SYN}/audioldm2/{g}')
print('real :', i.samplerate, i.channels, i.subtype, round(i.duration,2))
print('synth:', j.samplerate, j.channels, j.subtype, round(j.duration,2))

real : 16000 1 FLOAT 10.01
synth: 16000 1 FLOAT 10.0


In [5]:
import os, pandas as pd
SYN  = '/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw'
REAL = '/kaggle/working/real'
GENS = ['audioldm2','MusicGen_medium','musicldm','mustango','stable_audio_open']

usable = {os.path.splitext(f)[0] for f in os.listdir(REAL) if f.endswith('.wav')}

rows = [dict(path=f'{REAL}/{y}.wav', ytid=y, generator='real', label=0)
        for y in sorted(usable)]
for g in GENS:
    for f in sorted(os.listdir(f'{SYN}/{g}')):
        y = os.path.splitext(f)[0]
        if f.endswith('.wav') and y in usable:   # 169 captions have no authentic clip
            rows.append(dict(path=f'{SYN}/{g}/{f}', ytid=y, generator=g, label=1))

man = pd.DataFrame(rows)
print(man.shape)
print(man.groupby(['label','generator']).size())

(32112, 4)
label  generator        
0      real                 5352
1      MusicGen_medium      5352
       audioldm2            5352
       musicldm             5352
       mustango             5352
       stable_audio_open    5352
dtype: int64


In [6]:
import numpy as np
SEED = 42
rng = np.random.default_rng(SEED)

# Every ytid appears 6 times (1 real + 5 synthetic). Splitting on rows would
# put the same musical content in train and test, so we split on ytid.
ytids = np.array(sorted(man.ytid.unique()))
rng.shuffle(ytids)
n = len(ytids); n_tr, n_va = int(0.70*n), int(0.15*n)

split_of = {}
for y in ytids[:n_tr]:          split_of[y] = 'train'
for y in ytids[n_tr:n_tr+n_va]: split_of[y] = 'val'
for y in ytids[n_tr+n_va:]:     split_of[y] = 'test'

man['split'] = man.ytid.map(split_of)
man.to_csv('/kaggle/working/manifest.csv', index=False)
print(man.groupby(['split','label']).size())

split  label
test   0          804
       1         4020
train  0         3746
       1        18730
val    0          802
       1         4010
dtype: int64


In [7]:
sets = {s: set(man[man.split==s].ytid) for s in ['train','val','test']}
print('train & val :', len(sets['train'] & sets['val']))
print('train & test:', len(sets['train'] & sets['test']))
print('val   & test:', len(sets['val']   & sets['test']))

train & val : 0
train & test: 0
val   & test: 0


In [8]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret("kaggle_token")
print('token loaded')

token loaded


In [9]:
import shutil, os, json
shutil.make_archive('/kaggle/working/real_audio','zip','/kaggle/working/real')
os.makedirs('/kaggle/working/real_ds', exist_ok=True)
os.replace('/kaggle/working/real_audio.zip','/kaggle/working/real_ds/real_audio.zip')
json.dump({"title":"musiccaps-real-16k","id":"akshatyuvan/musiccaps-real-16k",
           "licenses":[{"name":"CC-BY-SA-4.0"}]},
          open('/kaggle/working/real_ds/dataset-metadata.json','w'))
!ls -lh /kaggle/working/real_ds

total 3.0G
-rw-r--r-- 1 root root  111 Oct  3 16:00 dataset-metadata.json
-rw-r--r-- 1 root root 3.0G Oct  3 16:00 real_audio.zip


In [10]:
!kaggle datasets create -p /kaggle/working/real_ds

Starting upload for file real_audio.zip
100%|███████████████████████████████████████| 2.94G/2.94G [00:13<00:00, 226MB/s]
Upload successful: real_audio.zip (3GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/akshatyuvan/musiccaps-real-16k


In [13]:
import json, os
os.makedirs('/kaggle/working/manifest_ds', exist_ok=True)
man.to_csv('/kaggle/working/manifest_ds/manifest.csv', index=False)
json.dump({"title":"fmc-manifest","id":"akshatyuvan/fmc-manifest",
           "licenses":[{"name":"CC0-1.0"}]},
          open('/kaggle/working/manifest_ds/dataset-metadata.json','w'))

In [14]:
!kaggle datasets version -p /kaggle/working/manifest_ds -m "add split column"

Starting upload for file manifest.csv
100%|██████████████████████████████████████| 3.21M/3.21M [00:00<00:00, 12.4MB/s]
Upload successful: manifest.csv (3MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/akshatyuvan/fmc-manifest
